# Experiments — FNO Kolmogorov Flow
Notebook reorganizado con las 4 secciones principales del experimento.

**Secciones:**
1. [Setup](#setup) — Imports, hiperparámetros, dataset, modelo
2. [Super-resolution](#super-resolution)
3. [Domain Decomposition](#domain-decomposition)
4. [Inverse Boundary Method (IBM)](#inverse-boundary-method)
5. [Fourier Spurious Attractor](#fourier-spurious-attractor)


## 0 · Setup <a id='setup'></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.ndimage import gaussian_filter
import copy

from utilities2 import (
    KolmogorovDataset,
    FNOGenerator,
    Rollout,
    SpectralConv2d,
    FNOBlock,
    NavierStokesResiduo,
    FNODiscriminatorStat,
    FNODiscriminatorPhys,
)


In [2]:
# ── Hiperparámetros ─────────────────────────────────────────
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
DATA_PATH = "../datasets/Snapshots/snapshots_64x64_use.npy"
H, W     = 64, 64
SEQ_LEN  = 10
BATCH    = 8
MODES    = 12
HIDDEN   = 32
N_LAYERS = 4
N_CRITIC = 2
Z_DIM    = 8
DT       = 0.05
LAMBDA_GP = 10.0

print(f"Device: {DEVICE}")


Device: cuda


In [3]:
# ── Dataset ─────────────────────────────────────────────────
from torch.utils.data import DataLoader, random_split

dataset   = KolmogorovDataset(DATA_PATH, seq_len=SEQ_LEN)
train_len = int(0.8 * len(dataset))
val_len   = len(dataset) - train_len
train_ds, val_ds = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(
    train_ds, batch_size=BATCH, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH, shuffle=False, num_workers=4,
)
print(f"Train: {train_len} | Val: {val_len}")


2026-04-07 16:49:09,442 | INFO | Dataset: 1152 trayectorias × 90 ventanas = 103,680 muestras | seq_len=10 | H×W=64×64


Train: 82944 | Val: 20736


In [4]:
# ── Cargar modelo ───────────────────────────────────────────
ckpt = torch.load("training_logs/best_checkpoint.pt", map_location="cpu")

G = FNOGenerator(hidden_ch=HIDDEN, modes1=MODES, modes2=MODES,
                 n_layers=N_LAYERS, z_dim=Z_DIM)
G.load_state_dict(ckpt["G"])
G.eval()

print(f"Checkpoint cargado — época {ckpt['epoch']}, val MSE={ckpt['val_mse']:.6f}")


Checkpoint cargado — época 27, val MSE=0.201850


In [5]:
# ── Condición inicial de referencia ─────────────────────────
idx          = val_loader.dataset.indices[0]
base_dataset = val_loader.dataset.dataset
traj_idx     = idx // base_dataset.n_windows

w_full = base_dataset.w[traj_idx]
w_true = torch.tensor(w_full).unsqueeze(0)   # (1, T, H, W)
w0     = w_true[:, 0].unsqueeze(1)           # (1, 1, H, W)
n_steps = w_true.shape[1] - 1

print(f"w_true shape: {w_true.shape} | w0 shape: {w0.shape} | n_steps: {n_steps}")


w_true shape: torch.Size([1, 100, 64, 64]) | w0 shape: torch.Size([1, 1, 64, 64]) | n_steps: 99


---
## 1 · Super-resolution <a id='super-resolution'></a>

El FNO fue entrenado a 64×64. Aquí se evalúa **zero-shot** a 256×256
interpolando bicúbicamente la condición inicial y dejando que el generador
opere en la resolución extendida.

El resultado se suaviza con un filtro gaussiano (σ=2) para atenuar
artefactos de alta frecuencia introducidos por la extrapolación espectral.


In [6]:
# ── Rollout LR (64×64) vs HR (256×256) ─────────────────────
w0_HR = F.interpolate(
    w0, size=(256, 256),
    mode="bicubic", align_corners=False
)  # (1, 1, 256, 256)

traj_LR = Rollout(G, device="cpu").run(w0,    n_steps=n_steps)
traj_HR = Rollout(G, device="cpu").run(w0_HR, n_steps=n_steps)

print(f"Shape LR: {traj_LR.shape}")
print(f"Shape HR: {traj_HR.shape}")

seq_LR = traj_LR[0].cpu().numpy()
seq_HR = traj_HR[0].cpu().numpy()
T      = min(seq_LR.shape[0], seq_HR.shape[0])

# Suavizado del HR para atenuar artefactos espectrales
seq_HR = np.stack([
    gaussian_filter(seq_HR[t], sigma=2.0)
    for t in range(T)
])


Shape LR: torch.Size([1, 100, 64, 64])
Shape HR: torch.Size([1, 100, 256, 256])


In [7]:
# ── Visualización: frames clave ─────────────────────────────
frames_of_interest = [0, 2, 4, 6, 8]
vmax = max(np.abs(seq_LR).max(), np.abs(seq_HR).max())
vmin = -vmax

fig, axes = plt.subplots(2, len(frames_of_interest), figsize=(20, 6))
fig.suptitle("Super-resolution — 64×64 vs 256×256 (zero-shot)", fontsize=13)

for j, t in enumerate(frames_of_interest):
    axes[0, j].imshow(seq_LR[t], cmap="RdBu_r", vmin=vmin, vmax=vmax)
    axes[0, j].set_title(f"LR t={t}", fontsize=8)
    axes[0, j].axis("off")

    axes[1, j].imshow(seq_HR[t], cmap="RdBu_r", vmin=vmin, vmax=vmax)
    axes[1, j].set_title(f"HR t={t}", fontsize=8)
    axes[1, j].axis("off")

plt.tight_layout()
plt.savefig("super_resolution_frames.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: super_resolution_frames.png")


Guardado: super_resolution_frames.png


/tmp/ipykernel_160288/2016548307.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── Visualización simple para reporte (sin zoom) ────────────

frames_of_interest = [0, 5]  # pocos frames para mejor tamaño

vmax = max(np.abs(seq_LR).max(), np.abs(seq_HR).max())
vmin = -vmax

for t in frames_of_interest:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(seq_LR[t], cmap="RdBu_r", vmin=vmin, vmax=vmax)
    axes[0].set_title(f"LR t={t}", fontsize=10)
    axes[0].axis("off")

    axes[1].imshow(seq_HR[t], cmap="RdBu_r", vmin=vmin, vmax=vmax)
    axes[1].set_title(f"HR t={t}", fontsize=10)
    axes[1].axis("off")

    plt.tight_layout()
    plt.savefig(f"super_res_t{t}.png", dpi=300, bbox_inches="tight")
    plt.close()

print("Figuras guardadas: super_res_t{t}.png")

Figuras guardadas: super_res_t{t}.png


In [11]:
# ── Animación: LR vs HR ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("FNO — Resolución original vs Super-resolución", fontsize=12)

axes[0].set_title("64×64")
axes[1].set_title("256×256 (zero-shot)")
for ax in axes:
    ax.axis("off")

im_LR = axes[0].imshow(seq_LR[0], cmap="RdBu_r", vmin=vmin, vmax=vmax)
im_HR = axes[1].imshow(seq_HR[0], cmap="RdBu_r", vmin=vmin, vmax=vmax)
fig.colorbar(im_LR, ax=axes.tolist(), shrink=0.6)
time_text = fig.text(0.5, 0.02, "t=0", ha="center")

def animate_sr(t):
    im_LR.set_data(seq_LR[t])
    im_HR.set_data(seq_HR[t])
    time_text.set_text(f"t={t}/{T-1}")
    return [im_LR, im_HR, time_text]

ani = animation.FuncAnimation(
    fig, animate_sr, frames=45, interval=100, blit=True
)
ani.save("super_resolution.gif", writer="pillow", dpi=120)
plt.close(fig)
print(f"Guardado: super_resolution.gif ({T} frames)")


2026-04-07 16:58:16,894 | INFO | Animation.save using <class 'matplotlib.animation.PillowWriter'>


Guardado: super_resolution.gif (100 frames)


---
## 2 · Domain Decomposition <a id='domain-decomposition'></a>

**Objetivo:** extender el dominio de 64×64 a 64×128 *sin reentrenar*,
aplicando descomposición de dominio con **Schwarz aditivo** y mezcla
coseno en la zona de overlap.

**Pipeline por paso:**
1. Extraer bloque izquierdo `[0:80]` y derecho `[48:128]` (overlap=16).
2. Propagar cada bloque a través del FNO.
3. Reconstruir el dominio completo con cosine blend en la zona `[64:80]`.

Se compara contra:
- Rollout base 64×64 (entrenamiento).
- Rollout 64×128 sin corrección (`RolloutRect`).


In [12]:
# ── Clase RolloutRect (referencia sin corrección) ───────────

class RolloutRect:
    """
    Rollout naive en dominio rectangular:
    el FNO procesa toda la imagen sin descomposición.
    Puede divergir en dominios > entrenamiento.
    """
    def __init__(self, generator, device="cpu"):
        self.G      = generator.to(device)
        self.device = device
        self.metrics = {}

    @torch.no_grad()
    def run(self, w0, n_steps, w_true=None):
        self.G.eval()
        w_cur = w0.to(self.device)
        traj  = [w_cur.squeeze(1).cpu()]

        for _ in range(n_steps):
            w_next = self.G(w_cur, z=None)
            traj.append(w_next.squeeze(1).cpu())
            w_cur = w_next

        traj = torch.stack(traj, dim=1)
        self._compute_metrics(traj, w_true)
        self.G.train()
        return traj

    def _compute_metrics(self, traj, w_true):
        self.metrics["energy"] = 0.5 * (traj**2).mean(dim=(0,2,3)).numpy()


In [13]:
# ── Clase RolloutDomainDecomp ────────────────────────────────

class RolloutDomainDecomp:
    """
    Zero-shot rollout en dominio extendido (H × W_ext) via
    domain decomposition con intercambio en frontera (Schwarz aditivo).

    Índices para W_train=64, overlap=16, W_ext=128:
      Bloque izq:    [0:80]
      Bloque der:    [48:128]
      Zona overlap:  [64:80]  (16 píxeles)
    """
    def __init__(self, generator, W_train=64, overlap=16, device="cpu"):
        self.G       = generator.to(device)
        self.W_train = W_train
        self.overlap = overlap
        self.device  = device
        self.metrics = {}

    def _cosine_blend(self, zone_left, zone_right, overlap):
        """Mezcla coseno: zone_left/right (B, H, overlap) → (B, H, overlap)."""
        t       = torch.linspace(0, 1, overlap, device=zone_left.device)
        w_left  = (0.5 * (1 + torch.cos(torch.pi * t))).view(1, 1, -1)
        w_right = (0.5 * (1 - torch.cos(torch.pi * t))).view(1, 1, -1)
        return w_left * zone_left + w_right * zone_right

    @torch.no_grad()
    def run(self, w0, n_steps, w_true=None):
        self.G.eval()
        B, _, H, W_ext = w0.shape
        W_train = self.W_train
        overlap = self.overlap

        l_start, l_end   = 0,               W_train + overlap   # 0,  80
        r_start, r_end   = W_train - overlap, W_ext              # 48, 128
        ov_start, ov_end = W_train,          W_train + overlap   # 64, 80
        ov_size          = ov_end - ov_start                     # 16
        ov_in_right      = ov_start - r_start                    # 16
        der_offset       = ov_end   - r_start                    # 32

        w_cur = w0.to(self.device)
        traj  = [w_cur.squeeze(1).cpu()]

        for _ in range(n_steps):
            w_left_next  = self.G(w_cur[:, :, :, l_start:l_end],  z=None)
            w_right_next = self.G(w_cur[:, :, :, r_start:r_end],  z=None)

            w_next = torch.zeros(B, 1, H, W_ext,
                                 device=self.device, dtype=w_cur.dtype)

            # Región izquierda pura
            w_next[:, :, :, :ov_start] = w_left_next[:, :, :, :ov_start]

            # Overlap — cosine blend (Schwarz aditivo)
            blend = self._cosine_blend(
                w_left_next[:, 0, :, -ov_size:],
                w_right_next[:, 0, :, ov_in_right:ov_in_right+ov_size],
                overlap=ov_size,
            )
            w_next[:, 0, :, ov_start:ov_end] = blend

            # Región derecha pura
            w_next[:, :, :, ov_end:] = w_right_next[:, :, :, der_offset:]

            traj.append(w_next.squeeze(1).cpu())
            w_cur = w_next

        traj = torch.stack(traj, dim=1)
        self._compute_metrics(traj, w_true)
        self.G.train()
        return traj

    def _compute_metrics(self, traj, w_true):
        self.metrics["energy"] = 0.5 * (traj**2).mean(dim=(0,2,3)).numpy()
        if w_true is not None:
            T    = min(traj.shape[1], w_true.shape[1])
            diff = traj[:, :T] - w_true[:, :T]
            self.metrics["rmse"] = (
                diff.pow(2).mean(dim=(0,2,3)).sqrt().numpy()
            )


In [14]:
# ── Funciones de diagnóstico ─────────────────────────────────

def energy_spectrum(field):
    fft    = np.fft.fft2(field)
    power  = np.abs(fft) ** 2
    H, W   = field.shape
    kx     = np.fft.fftfreq(H, d=1.0/H).astype(int)
    ky     = np.fft.fftfreq(W, d=1.0/W).astype(int)
    KX, KY = np.meshgrid(kx, ky, indexing="ij")
    K      = np.round(np.sqrt(KX**2 + KY**2)).astype(int)
    k_max  = min(H, W) // 2
    E      = np.array([power[K == k].mean() if (K == k).any() else 0
                       for k in range(k_max)])
    return np.arange(k_max), E

def energy_per_area(seq):
    return 0.5 * (seq ** 2).mean(axis=(1, 2))

def autocorrelation(field):
    f    = field - field.mean()
    fft  = np.fft.fft(f, axis=1)
    corr = np.real(np.fft.ifft(np.abs(fft)**2, axis=1)).mean(axis=0)
    return corr / (corr[0] + 1e-8)


In [15]:
# ── Experimento Domain Decomposition ────────────────────────
torch.manual_seed(42)
w0_sq   = torch.randn(1, 1, 64,  64)
w0_rect = torch.randn(1, 1, 64, 128)
n_steps_dd = 100

G_nofix = copy.deepcopy(G)

print("Rollout 64×64  (base)...")
traj_sq    = Rollout(G, device="cpu").run(w0_sq, n_steps=n_steps_dd)

print("Rollout 64×128 sin corrección...")
traj_nofix = RolloutRect(G_nofix, device="cpu").run(w0_rect, n_steps=n_steps_dd)

print("Rollout 64×128 domain decomp (overlap=16)...")
traj_dd    = RolloutDomainDecomp(G, W_train=64, overlap=16, device="cpu").run(
    w0_rect, n_steps=n_steps_dd
)

seq_sq    = traj_sq[0].numpy()
seq_nofix = traj_nofix[0].numpy()
seq_dd    = traj_dd[0].numpy()
T_dd      = min(seq_sq.shape[0], seq_nofix.shape[0], seq_dd.shape[0])
print("Listo.")


Rollout 64×64  (base)...
Rollout 64×128 sin corrección...
Rollout 64×128 domain decomp (overlap=16)...
Listo.


In [16]:
# ── Diagnósticos: espectro, energía, autocorrelación ────────
k_sq,    E_sq    = energy_spectrum(seq_sq[-1])
k_nofix, E_nofix = energy_spectrum(seq_nofix[-1])
k_dd,    E_dd    = energy_spectrum(seq_dd[-1])

ea_sq    = energy_per_area(seq_sq[:T_dd])
ea_nofix = energy_per_area(seq_nofix[:T_dd])
ea_dd    = energy_per_area(seq_dd[:T_dd])

ac_sq    = autocorrelation(seq_sq[-1])
ac_nofix = autocorrelation(seq_nofix[-1])
ac_dd    = autocorrelation(seq_dd[-1])

colors = ["steelblue", "tomato", "seagreen"]
labels = ["64×64", "64×128 sin corr.", "64×128 domain decomp"]
ls     = ["-", "--", "-."]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle("Domain Decomposition — Zero-Shot 64×128", fontsize=12)

ax = axes[0]
for k, E, c, lb, l in zip([k_sq, k_nofix, k_dd],
                            [E_sq, E_nofix, E_dd], colors, labels, ls):
    ax.loglog(k[1:], E[1:], label=lb, color=c, lw=2, ls=l)
ax.set_xlabel("Número de onda k"); ax.set_ylabel("E(k)")
ax.set_title("Espectro de Energía")
ax.legend(); ax.grid(True, which="both", alpha=0.3)

ax = axes[1]
for ea, c, lb, l in zip([ea_sq, ea_nofix, ea_dd], colors, labels, ls):
    ax.plot(ea, label=lb, color=c, lw=2, ls=l)
ax.set_xlabel("Paso t"); ax.set_ylabel("½‖w‖² / área")
ax.set_title("Energía por Unidad de Área")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
for ac, c, lb, l in zip([ac_sq, ac_nofix, ac_dd], colors, labels, ls):
    ax.plot(np.arange(len(ac)), ac, label=lb, color=c, lw=2, ls=l)
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.set_xlabel("Δx (píxeles)"); ax.set_ylabel("C(Δx)")
ax.set_title("Autocorrelación Espacial")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("diagnosticos_domain_decomp.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: diagnosticos_domain_decomp.png")


Guardado: diagnosticos_domain_decomp.png


/tmp/ipykernel_213802/3221183949.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# ── Animación Domain Decomposition ──────────────────────────
vmax_dd = max(np.abs(seq_sq).max(), np.abs(seq_nofix).max(), np.abs(seq_dd).max())
vmin_dd = -vmax_dd

fig, axes = plt.subplots(1, 3, figsize=(20, 5), gridspec_kw={"wspace": 0.04})
fig.suptitle("FNO — Domain Decomposition Zero-Shot", fontsize=12)

for ax, title in zip(axes, ["64×64 (train)", "64×128 sin corr.", "64×128 domain decomp"]):
    ax.set_title(title, fontsize=9)
    ax.axis("off")

im_sq    = axes[0].imshow(seq_sq[0],    cmap="RdBu_r", vmin=vmin_dd, vmax=vmax_dd)
im_nofix = axes[1].imshow(seq_nofix[0], cmap="RdBu_r", vmin=vmin_dd, vmax=vmax_dd)
im_dd    = axes[2].imshow(seq_dd[0],    cmap="RdBu_r", vmin=vmin_dd, vmax=vmax_dd)

fig.colorbar(im_sq, ax=axes.tolist(), shrink=0.7, label="ω (norm.)")
time_text = fig.text(0.5, 0.01, "t=0", ha="center", fontsize=11)

def animate_dd(t):
    im_sq.set_data(seq_sq[t])
    im_nofix.set_data(seq_nofix[t])
    im_dd.set_data(seq_dd[t])
    time_text.set_text(f"t={t}/{T_dd-1}")
    return [im_sq, im_nofix, im_dd, time_text]

ani = animation.FuncAnimation(fig, animate_dd, frames=45, interval=100, blit=True)
ani.save("zeroshot_domain_decomp.gif", writer="pillow", dpi=120)
plt.close(fig)
print(f"Guardado: zeroshot_domain_decomp.gif ({T_dd} pasos)")


2026-04-07 17:00:43,968 | INFO | Animation.save using <class 'matplotlib.animation.PillowWriter'>


Guardado: zeroshot_domain_decomp.gif (101 pasos)


---
## 3 · Inverse Boundary Method (IBM) <a id='inverse-boundary-method'></a>

Incorporación zero-shot de **obstáculos arbitrarios** vía penalización de
volumen. El pipeline por paso es:

```
w_t → [FNO] → w̃ → [Poisson FFT] → ψ̃ → [Vol. Pen.] → ψ_corr → [∇²ψ_corr] → w_{t+1}
```

Se comparan dos estrategias:
- **Hard mask:** `w_{t+1} = G(w_t) * mask`  (Dirichlet estricto).
- **IBM (Penalización de Volumen):** corrección suave a través del stream-function.

Las máscaras disponibles son: círculo y rectángulo.


In [21]:
# ═══════════════════════════════════════════════════════════════════════════
# FNO Zero-Shot IBM — Obstáculos Arbitrarios via Penalización de Volumen
# ═══════════════════════════════════════════════════════════════════════════
#
# Pipeline por paso:
#   w_t → [FNO] → w̃ → [Poisson FFT] → ψ̃ → [Vol. Pen. x10] → ψ_corr
#       → [∇²ψ_corr] → w_{t+1}
#
# Requiere: torch, numpy, scipy
# Compatible con RolloutMasked existente — misma interfaz .run()
# ═══════════════════════════════════════════════════════════════════════════

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.ndimage import distance_transform_edt


# ───────────────────────────────────────────────────────────────────────────
# 1. SDF — Signed Distance Function desde máscara arbitraria
# ───────────────────────────────────────────────────────────────────────────

def compute_sdf(mask_np: np.ndarray) -> np.ndarray:
    """
    Dado mask_np (H, W) binaria — 0 dentro del obstáculo, 1 fuera —
    retorna SDF (H, W): positivo fuera, negativo dentro, 0 en frontera.
    """
    dist_out = distance_transform_edt(mask_np)        # distancia desde fuera
    dist_in  = distance_transform_edt(1 - mask_np)    # distancia desde dentro
    sdf      = dist_out - dist_in
    return sdf


def make_circle_mask(H: int, W: int,
                     cx: float = 0.5, cy: float = 0.5,
                     radius: float = 0.1) -> np.ndarray:
    """
    Máscara binaria circular. Retorna (H, W) numpy — 0 dentro, 1 fuera.
    Coordenadas normalizadas [0,1].
    """
    xs = np.linspace(0, 1, W)
    ys = np.linspace(0, 1, H)
    YY, XX = np.meshgrid(ys, xs, indexing="ij")
    dist = np.sqrt((XX - cx)**2 + (YY - cy)**2)
    return (dist > radius).astype(np.float32)


def make_rectangle_mask(H: int, W: int,
                        x0: float = 0.4, y0: float = 0.4,
                        x1: float = 0.6, y1: float = 0.6) -> np.ndarray:
    """
    Máscara binaria rectangular. Retorna (H, W) numpy — 0 dentro, 1 fuera.
    Coordenadas normalizadas [0,1].
    """
    xs = np.linspace(0, 1, W)
    ys = np.linspace(0, 1, H)
    YY, XX = np.meshgrid(ys, xs, indexing="ij")
    inside = (XX >= x0) & (XX <= x1) & (YY >= y0) & (YY <= y1)
    return (~inside).astype(np.float32)


def make_arbitrary_mask(H: int, W: int,
                        mask_np: np.ndarray) -> np.ndarray:
    """
    Acepta cualquier máscara binaria (H, W) — 0 dentro, 1 fuera.
    Útil para formas cargadas externamente.
    """
    assert mask_np.shape == (H, W), "La máscara debe ser (H, W)"
    return mask_np.astype(np.float32)


# ───────────────────────────────────────────────────────────────────────────
# 2. SOLVER DE POISSON FFT — dominio periódico cuadrado
# ───────────────────────────────────────────────────────────────────────────

def build_poisson_kernel(H: int, W: int, device: str = "cpu") -> torch.Tensor:
    """
    Precomputa el kernel espectral para ∇²ψ = -w en dominio periódico.
    k² = kx² + ky²  (frecuencias modales para grilla HxW con dominio [0,2π])
    Retorna (H, W) tensor con 1/k² — el modo (0,0) se pone a 0 (gauge).
    """
    kx = torch.fft.fftfreq(W, d=1.0/W).to(device)  # frecuencias enteras
    ky = torch.fft.fftfreq(H, d=1.0/H).to(device)
    KY, KX = torch.meshgrid(ky, kx, indexing="ij")
    k2 = KX**2 + KY**2
    k2[0, 0] = 1.0          # evitar división por cero; se corrige abajo
    inv_k2 = 1.0 / k2
    inv_k2[0, 0] = 0.0      # gauge: ψ de media cero
    return inv_k2            # (H, W)


def poisson_fft(w: torch.Tensor, inv_k2: torch.Tensor) -> torch.Tensor:
    """
    Resuelve ∇²ψ = -w con FFT.
    w       : (B, 1, H, W)
    inv_k2  : (H, W)
    Retorna ψ: (B, 1, H, W)
    """
    w_hat   = torch.fft.fft2(w.squeeze(1))              # (B, H, W) complejo
    psi_hat = -w_hat * inv_k2.unsqueeze(0)             # ψ̂ = -ŵ / k²  (∇²ψ = -w)
    psi     = torch.fft.ifft2(psi_hat).real            # (B, H, W)
    return psi.unsqueeze(1)                            # (B, 1, H, W)


def recover_vorticity(psi: torch.Tensor, k2: torch.Tensor) -> torch.Tensor:
    """
    Recupera w = -∇²ψ via FFT.
    psi : (B, 1, H, W)
    k2  : (H, W)
    Retorna w: (B, 1, H, W)
    """
    psi_hat = torch.fft.fft2(psi.squeeze(1))
    w_hat   = -psi_hat * k2.unsqueeze(0)
    w       = torch.fft.ifft2(w_hat).real
    return w.unsqueeze(1)


# ───────────────────────────────────────────────────────────────────────────
# 3. PENALIZACIÓN DE VOLUMEN — impone ψ=0 dentro del obstáculo
# ───────────────────────────────────────────────────────────────────────────

def volume_penalization(
    psi: torch.Tensor,
    chi: torch.Tensor,
    inv_k2: torch.Tensor,
    eta: float = 0.05,
    n_iter: int = 10,
    psi_target: float = 0.0,
    omega: float = 0.2,
) -> torch.Tensor:
    """
    Itera la corrección de ψ para imponer ψ=psi_target dentro del obstáculo.

    En cada iteración:
        f    = chi * (psi_target - ψ) / η     ← fuerza de penalización
        dpsi = Poisson_FFT(f)                  ← corrección espectral
        ψ   ← ψ + ω * dpsi                    ← relajación para estabilidad

    chi  : (1, 1, H, W) — indicador suave basado en SDF.
           1 dentro del obstáculo, 0 fuera, transición de δ píxeles.
    psi  : (B, 1, H, W)
    Retorna ψ corregido: (B, 1, H, W)
    """
    for _ in range(n_iter):
        penalty  = chi * (psi_target - psi) / eta
        pen_hat  = torch.fft.fft2(penalty.squeeze(1))
        dpsi_hat = pen_hat * inv_k2.unsqueeze(0)
        dpsi     = torch.fft.ifft2(dpsi_hat).real.unsqueeze(1)
        psi      = psi + omega * dpsi

    return psi


# ───────────────────────────────────────────────────────────────────────────
# 4. ROLLOUT IBM — integra FNO + corrección física
# ───────────────────────────────────────────────────────────────────────────

class RolloutIBM:
    """
    Rollout zero-shot con obstáculos arbitrarios via penalización de volumen.

    Uso:
        mask_np = make_circle_mask(64, 64, cx=0.5, cy=0.5, radius=0.1)
        rollout = RolloutIBM(G, mask_np, eta=0.05, n_iter=10, device="cpu")
        traj    = rollout.run(w0, n_steps=100)

    Compatible con cualquier geometría — circular, rectangular, arbitraria.
    """

    def __init__(self,
                 generator,
                 mask_np: np.ndarray,
                 eta: float = 0.05,
                 n_iter: int = 10,
                 omega: float = 0.2,
                 delta: float = 1.5,
                 energy_threshold: float = 1e6,
                 device: str = "cpu"):
        """
        delta : ancho de la zona de transición en píxeles.
                1.0-2.0 → frontera nítida, concentrada en el límite.
                5.0+    → transición suave (comportamiento anterior).
        """

        self.G                = generator.to(device)
        self.eta              = eta
        self.n_iter           = n_iter
        self.omega            = omega
        self.energy_threshold = energy_threshold
        self.device           = device
        self.metrics          = {}

        H, W = mask_np.shape

        # Máscara binaria como tensor (para contornos y referencia)
        self.obstacle = torch.tensor(mask_np, dtype=torch.float32,
                                     device=device).view(1, 1, H, W)

        # SDF en píxeles: positivo fuera, negativo dentro
        sdf_np   = compute_sdf(mask_np)
        self.sdf = sdf_np

        # chi suave basada en SDF:
        #   sigmoid(-sdf/delta) → 1 dentro del obstáculo, 0 fuera
        #   delta controla el ancho de la transición en píxeles
        sdf_t    = torch.tensor(sdf_np, dtype=torch.float32, device=device)
        self.chi = torch.sigmoid(-sdf_t / delta).view(1, 1, H, W)

        # Kernel de Poisson precomputado
        kx = torch.fft.fftfreq(W, d=1.0/W).to(device)
        ky = torch.fft.fftfreq(H, d=1.0/H).to(device)
        KY, KX = torch.meshgrid(ky, kx, indexing="ij")
        self.k2 = KX**2 + KY**2

        inv_k2 = self.k2.clone()
        inv_k2[0, 0] = 1.0
        self.inv_k2 = 1.0 / inv_k2
        self.inv_k2[0, 0] = 0.0

    @torch.no_grad()
    def run(self, w0: torch.Tensor, n_steps: int,
            w_true: torch.Tensor = None) -> torch.Tensor:
        """
        w0     : (B, 1, H, W) — condición inicial
        n_steps: número de pasos autorregresivos
        w_true : (B, T, H, W) opcional para métricas
        Retorna traj: (B, T+1, H, W)
        """
        self.G.eval()
        w_cur = w0.to(self.device)
        traj  = [w_cur.squeeze(1).cpu()]

        for step in range(n_steps):
            # 1. FNO predice flujo libre
            w_pred = self.G(w_cur, z=None)

            # 2. Poisson FFT → stream function
            psi = poisson_fft(w_pred, self.inv_k2)

            # 3. Penalización de volumen con relajación ω
            psi_corr = volume_penalization(
                psi, self.chi, self.inv_k2,
                eta=self.eta, n_iter=self.n_iter, omega=self.omega
            )

            # 4. Recuperar vorticidad corregida
            w_next = recover_vorticity(psi_corr, self.k2)

            # 5. Chequeo de divergencia
            energy = 0.5 * (w_next**2).mean().item()
            if energy > self.energy_threshold:
                print(f"  ⚠️  Divergencia detectada en paso t={step} "
                      f"(energía={energy:.2e} > umbral={self.energy_threshold:.1e}). "
                      f"Intenta subir omega o reducir eta.")
                traj = torch.stack(traj, dim=1)
                self._compute_metrics(traj, w_true)
                self.G.train()
                return traj

            traj.append(w_next.squeeze(1).cpu())
            w_cur = w_next

        traj = torch.stack(traj, dim=1)   # (B, T+1, H, W)
        self._compute_metrics(traj, w_true)
        self.G.train()
        return traj

    def _compute_metrics(self, traj: torch.Tensor,
                         w_true: torch.Tensor = None):
        """Energía por paso y RMSE si hay referencia."""
        self.metrics["energy"] = 0.5 * (traj**2).mean(dim=(0, 2, 3)).numpy()

        if w_true is not None:
            T    = min(traj.shape[1], w_true.shape[1])
            diff = traj[:, :T] - w_true[:, :T]
            self.metrics["rmse"] = (
                diff.pow(2).mean(dim=(0, 2, 3)).sqrt().numpy()
            )


# ───────────────────────────────────────────────────────────────────────────
# 5. EXPERIMENTO — comparación Máscara Hard vs IBM Zero-Shot
# ───────────────────────────────────────────────────────────────────────────

def run_experiment(G, w0, val_loader=None, w_true=None,
                   H=64, W=64, n_steps=100):
    """
    Corre tres configuraciones de obstáculos con IBM y las compara
    contra la máscara hard original y el rollout limpio.

    G         : modelo FNO ya cargado
    w0        : (1, 1, H, W) condición inicial
    w_true    : (1, T, H, W) trayectoria de referencia (opcional)
    """

    configs = [
        {
            "mask_fn": lambda: make_circle_mask(H, W, cx=0.5, cy=0.5, radius=0.08),
            "label": "Círculo centro r=0.08"
        },
        {
            "mask_fn": lambda: make_circle_mask(H, W, cx=0.3, cy=0.4, radius=0.12),
            "label": "Círculo offset r=0.12"
        },
        {
            "mask_fn": lambda: make_rectangle_mask(H, W, x0=0.4, y0=0.4,
                                                    x1=0.6, y1=0.6),
            "label": "Rectángulo centro"
        },
    ]

    print("Corriendo experimento IBM zero-shot...\n")

    results = []
    for cfg in configs:
        mask_np = cfg["mask_fn"]()
        print(f"  → {cfg['label']}")

        rollout = RolloutIBM(G, mask_np, eta=0.05, n_iter=10,
                             omega=0.2, delta=1,
                             energy_threshold=1e6, device="cpu")
        traj    = rollout.run(w0, n_steps=n_steps, w_true=w_true)

        results.append({
            "seq":     traj[0].numpy(),      # (T+1, H, W)
            "mask":    mask_np,
            "label":   cfg["label"],
            "energy":  rollout.metrics["energy"],
            "rmse":    rollout.metrics.get("rmse", None),
        })

    # ── Diagnóstico: energía por paso ──────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 4))
    colors  = ["tomato", "seagreen", "darkorchid"]

    for r, c in zip(results, colors):
        ax.plot(r["energy"], label=f"IBM — {r['label']}", color=c, lw=2)

    ax.set_xlabel("Paso t")
    ax.set_ylabel("½‖w‖²  promedio")
    ax.set_title("Energía por Paso — IBM Zero-Shot vs Máscara Hard")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("energia_ibm_zeroshot.png", dpi=150)
    plt.show()
    print("\nGuardado: energia_ibm_zeroshot.png")

    # ── Animación: 3 obstáculos IBM ────────────────────────────────────────
    T    = min(r["seq"].shape[0] for r in results)

    # vmin/vmax globales fijos — calculados UNA sola vez antes de animar
    vmax = float(max(np.abs(r["seq"][:T]).max() for r in results))
    vmin = -vmax

    fig, axes = plt.subplots(1, 3, figsize=(18, 5),
                             gridspec_kw={"wspace": 0.08})
    fig.suptitle("FNO Zero-Shot IBM — Penalización de Volumen", fontsize=12)

    ims = []
    for ax, r in zip(axes, results):
        ax.set_title(r["label"], fontsize=9)
        ax.axis("off")
        # interpolation="none" evita artefactos visuales entre frames
        im = ax.imshow(r["seq"][0], cmap="RdBu_r",
                       vmin=vmin, vmax=vmax,
                       interpolation="none",
                       animated=True)
        # Contorno estático del obstáculo — se dibuja una sola vez
        chi = 1.0 - r["mask"]
        ax.contour(chi, levels=[0.5], colors=["black"], linewidths=1.5)
        ims.append(im)

    # Colorbar estático fuera del loop de animación
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    fig.colorbar(ims[0], cax=cbar_ax, label="ω (norm.)")

    # Texto de tiempo como artista de axes (más estable que fig.text con blit)
    time_text = axes[1].set_title(f"{results[1]['label']}\nt=0", fontsize=9)

    def animate(t):
        for i, (im, r) in enumerate(zip(ims, results)):
            im.set_data(r["seq"][t])
        # Actualizar título del panel central con el tiempo
        axes[1].set_title(f"{results[1]['label']}\nt={t}/{T-1}", fontsize=9)
        return ims  # blit=False — el return es ignorado pero requerido

    ani = animation.FuncAnimation(fig, animate, frames=T,
                                  interval=80, blit=False)
    ani.save("ibm_zeroshot.gif", writer="pillow", dpi=110)
    plt.close(fig)
    print(f"Guardado: ibm_zeroshot.gif  ({T} pasos)")

    return results


# ───────────────────────────────────────────────────────────────────────────
# 6. PUNTO DE ENTRADA — pega esto en tu notebook después de tener G y w0
# ───────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # ── Esto asume que ya tienes G, val_loader definidos en tu entorno ──
    # Descomenta y adapta:

    # idx          = val_loader.dataset.indices[0]
    # base_dataset = val_loader.dataset.dataset
    # traj_idx     = idx // base_dataset.n_windows
    # w_full       = base_dataset.w[traj_idx]
    # w_true       = torch.tensor(w_full).unsqueeze(0)   # (1, T, H, W)
    # w0           = w_true[:, 0].unsqueeze(1)            # (1, 1, H, W)

    results = run_experiment(G, w0, w_true=w_true, H=64, W=64, n_steps=100)


Corriendo experimento IBM zero-shot...

  → Círculo centro r=0.08
  → Círculo offset r=0.12
  → Rectángulo centro


/tmp/ipykernel_213802/699595981.py:348: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-07 17:06:31,833 | INFO | Animation.save using <class 'matplotlib.animation.PillowWriter'>



Guardado: energia_ibm_zeroshot.png
Guardado: ibm_zeroshot.gif  (101 pasos)


---
## 4 · Fourier Spurious Attractor <a id='fourier-spurious-attractor'></a>

El generador FNO puede converger a un **atractor espurio** en el espacio
espectral: el rollout autoregresivo sin corrección tiende a colapsar a
un estado de baja energía o a divergir.

Aquí se estudia este fenómeno a través de:
1. **Análisis espectral** por bandas (baja / media / alta frecuencia).
2. **Rollout normalizado** (corrección de std en cada paso) como mitigación.
3. **Fréchet Distance** física para cuantificar la calidad del atractor
   respecto al Ground Truth.


In [22]:
# ── Análisis espectral por bandas ───────────────────────────
import torch.fft as fft

traj_base = Rollout(G, device="cpu").run(w0, n_steps=n_steps)
seq_base  = traj_base[0].cpu().numpy()

kx = fft.fftfreq(64, d=1.0/64)
ky = fft.rfftfreq(64, d=1.0/64)
KX, KY = torch.meshgrid(kx, ky, indexing="ij")
K = torch.sqrt(KX**2 + KY**2)

low_ratio, mid_ratio, high_ratio = [], [], []

for t in range(seq_base.shape[0]):
    frame  = torch.tensor(seq_base[t]).unsqueeze(0).unsqueeze(0)
    w_hat  = fft.rfft2(frame)
    energy = (torch.abs(w_hat)**2).squeeze()

    e_low  = energy[K <= 10].sum().item()
    e_mid  = energy[(K > 10) & (K <= 25)].sum().item()
    e_high = energy[K > 25].sum().item()
    total  = e_low + e_mid + e_high + 1e-12

    low_ratio.append(e_low / total)
    mid_ratio.append(e_mid / total)
    high_ratio.append(e_high / total)

steps = np.arange(len(low_ratio))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, low_ratio,  label="Baja  (k≤10)",   color="steelblue", lw=2)
ax.plot(steps, mid_ratio,  label="Media (10<k≤25)", color="seagreen",  lw=2)
ax.plot(steps, high_ratio, label="Alta  (k>25)",    color="tomato",    lw=2)
ax.set_xlabel("Paso t"); ax.set_ylabel("Fracción de energía")
ax.set_title("Distribución espectral de energía — atractor espurio")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("espectro_bandas.png", dpi=150)
plt.show()
print("Guardado: espectro_bandas.png")


Guardado: espectro_bandas.png


/tmp/ipykernel_213802/2448016039.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
# ── Rollout normalizado (mitigación del atractor) ────────────

@torch.no_grad()
def normalized_rollout(generator, w0, n_steps, device="cpu"):
    """
    Normalización por std en cada paso para evitar colapso espectral.
    Rescala w_{t+1} a la std del estado inicial.
    """
    generator.eval()
    w_cur = w0.to(device)
    std_0 = w_cur.std()

    traj = [w_cur.squeeze(1).cpu()]
    for _ in range(n_steps):
        w_next = generator(w_cur)
        std_cur = w_next.std()
        w_next  = w_next * (std_0 / (std_cur + 1e-8))
        traj.append(w_next.squeeze(1).cpu())
        w_cur = w_next

    generator.train()
    return torch.stack(traj, dim=1)


traj_norm = normalized_rollout(G, w0, n_steps=n_steps, device="cpu")
seq_norm  = traj_norm[0].cpu().numpy()
print(f"Rollout normalizado: {traj_norm.shape}")


Rollout normalizado: torch.Size([1, 100, 64, 64])


In [24]:
# ── Frames clave: base vs normalizado ───────────────────────
frames_key = [0, int(n_steps*0.3), int(n_steps*0.6), int(n_steps*0.75),
              int(n_steps*0.9), n_steps-1]

seq_gt = w_true[0].cpu().numpy()
vmax_fa = max(np.abs(seq_gt).max(), np.abs(seq_base).max(), np.abs(seq_norm).max())
vmin_fa = -vmax_fa

fig, axes = plt.subplots(3, len(frames_key), figsize=(22, 7))
fig.suptitle("Atractor Espurio — Ground Truth vs Base vs Normalizado", fontsize=12)
row_labels = ["Ground Truth", "Base (diverge)", "Normalizado"]

for row, (seq, rl) in enumerate(zip([seq_gt, seq_base, seq_norm], row_labels)):
    for j, t in enumerate(frames_key):
        t_clamp = min(t, seq.shape[0]-1)
        axes[row, j].imshow(seq[t_clamp], cmap="RdBu_r", vmin=vmin_fa, vmax=vmax_fa)
        axes[row, j].set_title(f"t={t}", fontsize=8)
        axes[row, j].axis("off")
    axes[row, 0].set_ylabel(rl, fontsize=9, rotation=90, va="center")

plt.tight_layout()
plt.savefig("atractor_comparacion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: atractor_comparacion.png")


Guardado: atractor_comparacion.png


/tmp/ipykernel_213802/3531830741.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
# ── Rollout largo (1000 pasos) — ¿converge a un atractor? ───
traj_long = normalized_rollout(G, w0, n_steps=1000, device="cpu")
seq_long  = traj_long[0].cpu().numpy()

frames_long = [0, 100, 200, 400, 600, 800, 999]
fig, axes = plt.subplots(1, len(frames_long), figsize=(24, 4))
fig.suptitle("Rollout normalizado — 1000 pasos (atractor de largo plazo)", fontsize=12)

vmax_long = np.abs(seq_long).max()
for ax, t in zip(axes, frames_long):
    ax.imshow(seq_long[t], cmap="RdBu_r", vmin=-vmax_long, vmax=vmax_long)
    ax.set_title(f"t={t}", fontsize=9)
    ax.axis("off")

fig.colorbar(axes[-1].images[0], ax=axes.tolist(), shrink=0.6, label="ω (norm.)")
plt.savefig("atractor_largo_plazo.png", dpi=120, bbox_inches="tight")
plt.close()
print("Guardado: atractor_largo_plazo.png")


Guardado: atractor_largo_plazo.png


### 4.1 · Fréchet Distance física

Se usa un encoder que proyecta trayectorias al espacio de features
(estadístico + físico vía residuo Navier-Stokes) y se calcula la
distancia de Fréchet entre distribuciones.

Cuatro condiciones evaluadas:
- GT vs GT (piso mínimo)
- GT vs Generado (base)
- GT vs Generado+Norm
- GT vs Rand (techo)


In [32]:
# ── Cargar discriminadores y construir extractor ─────────────
ckpt_v1 = torch.load("training_logs/best_checkpoint.pt", map_location="cpu")

G_fd = FNOGenerator(hidden_ch=32, modes1=12, modes2=12, n_layers=4, z_dim=8)
G_fd.load_state_dict(ckpt_v1["G"])
G_fd.eval()

D_stat = FNODiscriminatorStat(
    seq_len=SEQ_LEN + 1, hidden_ch=HIDDEN,
    modes1=MODES, modes2=MODES, n_layers=N_LAYERS,
)
D_stat.load_state_dict(ckpt["D_stat"])
D_stat.eval()

D_phys = FNODiscriminatorPhys(
    seq_len=SEQ_LEN + 1, hidden_ch=HIDDEN,
    modes1=MODES, modes2=MODES, n_layers=N_LAYERS,
)
D_phys.load_state_dict(ckpt["D_phys"])
D_phys.eval()

print(f"G_fd cargado — época {ckpt_v1['epoch']}, val MSE={ckpt_v1['val_mse']:.6f}")
print(f"Discriminadores cargados — época {ckpt['epoch']}")


G_fd cargado — época 27, val MSE=0.201850
Discriminadores cargados — época 27


In [33]:
# ── FrechetFeatureExtractor ──────────────────────────────────
from scipy import linalg as sp_linalg

class FrechetFeatureExtractor:
    def __init__(self, d_stat, d_phys, ns_residuo, device="cpu"):
        self.d_stat     = d_stat.to(device).eval()
        self.d_phys     = d_phys.to(device).eval()
        self.ns_residuo = ns_residuo.to(device)
        self.device     = device

    def _features_stat(self, traj):
        """traj: (B, T, H, W) → (B, 32)"""
        B, T, H, W = traj.shape
        x = traj.permute(0, 2, 3, 1).contiguous().reshape(B * H * W, T, 1)
        x = self.d_stat.temporal_mix(x).squeeze(-1)
        x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2)
        for layer in self.d_stat.layers:
            x = layer(x)
        return x.mean(dim=(-2, -1))

    def _features_phys(self, traj):
        """traj: (B, T, H, W) → (B, 32)"""
        residuo    = self.ns_residuo.residuo_espacial(traj)
        B, T, H, W = residuo.shape
        x = residuo.permute(0, 2, 3, 1).contiguous().reshape(B * H * W, T, 1)
        x = self.d_phys.temporal_mix(x).squeeze(-1)
        x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2)
        for layer in self.d_phys.layers:
            x = layer(x)
        return x.mean(dim=(-2, -1))

    @torch.no_grad()
    def extract(self, loader, n_samples=None, use_generated=False,
                generator=None, normalized=False):
        if generator is not None:
            generator = generator.to(self.device).eval()
        features = []
        count    = 0

        for seq_in, _, real_traj in loader:
            real_traj = real_traj.to(self.device)

            if use_generated:
                B, T, H, W = real_traj.shape
                w_cur = real_traj[:, 0].unsqueeze(1)
                fake  = [w_cur.squeeze(1)]
                for _ in range(T - 1):
                    w_next = generator(w_cur)
                    if normalized:
                        w_next = w_next / (w_next.std() + 1e-8)
                    fake.append(w_next.squeeze(1))
                    w_cur = w_next
                traj = torch.stack(fake, dim=1)
            else:
                traj = real_traj

            f = torch.cat([self._features_stat(traj),
                           self._features_phys(traj)], dim=1)
            features.append(f.cpu())
            count += traj.shape[0]
            if n_samples is not None and count >= n_samples:
                break

        return torch.cat(features, dim=0)[:n_samples].numpy()


def frechet_distance(feat_real, feat_fake):
    mu_r  = feat_real.mean(axis=0)
    mu_f  = feat_fake.mean(axis=0)
    sig_r = np.cov(feat_real, rowvar=False)
    sig_f = np.cov(feat_fake, rowvar=False)
    diff  = mu_r - mu_f
    covmean, _ = sp_linalg.sqrtm(sig_r @ sig_f, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(sig_r + sig_f - 2 * covmean))


In [34]:
# ── Pipeline de features con sampling aleatorio ──────────────
from torch.utils.data import SubsetRandomSampler

ns_residuo = NavierStokesResiduo(H=64, W=64, dt=0.05,
                                  modes1=12, modes2=12, device=DEVICE).to(DEVICE)
extractor  = FrechetFeatureExtractor(D_stat, D_phys, ns_residuo, device=DEVICE)


def extract_features_random(extractor, dataset, n_samples=100,
                             use_generated=False, generator=None,
                             normalized=False, batch_size=8):
    """Samplea aleatoriamente n_samples del dataset en cada llamada."""
    indices = np.random.choice(len(dataset), size=n_samples, replace=False)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         sampler=SubsetRandomSampler(indices))
    return extractor.extract(loader, n_samples=n_samples,
                             use_generated=use_generated,
                             generator=generator, normalized=normalized)


In [35]:
# ── Cálculo de Fréchet Distance (10 repeticiones) ───────────
n_fd = 100

print("Calculando variabilidad natural GT (piso)...")
fd_gt_list = []
for i in range(10):
    fa = extract_features_random(extractor, val_ds, n_samples=n_fd)
    fb = extract_features_random(extractor, val_ds, n_samples=n_fd)
    fd_gt_list.append(frechet_distance(fa, fb))

print("Calculando GT vs Generado (base)...")
fd_gen_list = []
for i in range(10):
    fa = extract_features_random(extractor, val_ds, n_samples=n_fd)
    fb = extract_features_random(extractor, val_ds, n_samples=n_fd,
                                  use_generated=True, generator=G_fd)
    fd_gen_list.append(frechet_distance(fa, fb))

print("Calculando GT vs Generado+Norm...")
fd_norm_list = []
for i in range(10):
    fa = extract_features_random(extractor, val_ds, n_samples=n_fd)
    fb = extract_features_random(extractor, val_ds, n_samples=n_fd,
                                  use_generated=True, generator=G_fd, normalized=True)
    fd_norm_list.append(frechet_distance(fa, fb))

print("Calculando GT vs Rand (techo)...")
fd_rand_list = []
for i in range(10):
    fa = extract_features_random(extractor, val_ds, n_samples=n_fd)
    # Rand: rollout desde ruido puro
    indices_r = np.random.choice(len(val_ds), size=n_fd, replace=False)
    loader_r  = DataLoader(val_ds, batch_size=8, sampler=SubsetRandomSampler(indices_r))
    feats_rand = []
    G_fd.eval()
    with torch.no_grad():
        for seq_in, _, real_traj in loader_r:
            real_traj = real_traj.to(DEVICE)
            B, T, H_t, W_t = real_traj.shape
            w_cur = torch.randn(B, 1, H_t, W_t, device=DEVICE)
            fake  = [w_cur.squeeze(1)]
            for _ in range(T - 1):
                w_next = G_fd.to(DEVICE)(w_cur)
                w_next = w_next / (w_next.std() + 1e-8)
                fake.append(w_next.squeeze(1))
                w_cur = w_next
            traj = torch.stack(fake, dim=1)
            f = torch.cat([extractor._features_stat(traj),
                           extractor._features_phys(traj)], dim=1)
            feats_rand.append(f.cpu())
    fb = torch.cat(feats_rand, dim=0)[:n_fd].numpy()
    fd_rand_list.append(frechet_distance(fa, fb))

print(f"\n{'='*55}")
print(f"FD GT vs GT:   {np.mean(fd_gt_list):.4f} ± {np.std(fd_gt_list):.4f}  ← piso")
print(f"FD GT vs Gen:  {np.mean(fd_gen_list):.4f} ± {np.std(fd_gen_list):.4f}")
print(f"FD GT vs Norm: {np.mean(fd_norm_list):.4f} ± {np.std(fd_norm_list):.4f}")
print(f"FD GT vs Rand: {np.mean(fd_rand_list):.4f} ± {np.std(fd_rand_list):.4f}  ← techo")
print(f"{'='*55}")


Calculando variabilidad natural GT (piso)...
Calculando GT vs Generado (base)...
Calculando GT vs Generado+Norm...
Calculando GT vs Rand (techo)...

FD GT vs GT:   0.0027 ± 0.0012  ← piso
FD GT vs Gen:  0.2461 ± 0.0123
FD GT vs Norm: 0.2649 ± 0.0229
FD GT vs Rand: 0.5027 ± 0.0168  ← techo


In [36]:
# ── Visualización PCA y t-SNE del espacio de features ───────
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

print("Extrayendo features para visualización (200 muestras por clase)...")
feat_gt   = extract_features_random(extractor, val_ds, n_samples=200)
feat_gen  = extract_features_random(extractor, val_ds, n_samples=200,
                                     use_generated=True, generator=G_fd)
feat_norm = extract_features_random(extractor, val_ds, n_samples=200,
                                     use_generated=True, generator=G_fd, normalized=True)

# Rand (200 muestras)
G_fd.eval()
feats_rand_vis = []
with torch.no_grad():
    indices_rv = np.random.choice(len(val_ds), size=200, replace=False)
    loader_rv  = DataLoader(val_ds, batch_size=8, sampler=SubsetRandomSampler(indices_rv))
    for seq_in, _, real_traj in loader_rv:
        real_traj = real_traj.to(DEVICE)
        B, T, H_t, W_t = real_traj.shape
        w_cur = torch.randn(B, 1, H_t, W_t, device=DEVICE)
        fake  = [w_cur.squeeze(1)]
        for _ in range(T - 1):
            w_next = G_fd.to(DEVICE)(w_cur)
            w_next = w_next / (w_next.std() + 1e-8)
            fake.append(w_next.squeeze(1))
            w_cur = w_next
        traj = torch.stack(fake, dim=1)
        f = torch.cat([extractor._features_stat(traj),
                       extractor._features_phys(traj)], dim=1)
        feats_rand_vis.append(f.cpu())
feat_rand_vis = torch.cat(feats_rand_vis, dim=0)[:200].numpy()

all_feats = np.concatenate([feat_gt, feat_gen, feat_norm, feat_rand_vis], axis=0)
labels    = (["Ground Truth"]*200 + ["Generado"]*200 +
             ["Gen + Norm"]*200   + ["Rand"]*200)
col_map   = {"Ground Truth": "tab:blue", "Generado": "tab:red",
             "Gen + Norm":   "tab:green", "Rand": "tab:orange"}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Espacio de features — Fréchet (Atractor Espurio)", fontsize=12)

# PCA
print("Calculando PCA...")
pca    = PCA(n_components=2)
pca_2d = pca.fit_transform(all_feats)
for lbl, col in col_map.items():
    mask = np.array(labels) == lbl
    axes[0].scatter(pca_2d[mask, 0], pca_2d[mask, 1],
                    label=lbl, color=col, alpha=0.6, s=20)
axes[0].set_title(
    f"PCA  (PC1={pca.explained_variance_ratio_[0]*100:.1f}%, "
    f"PC2={pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=11)
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# t-SNE
print("Calculando t-SNE (puede tardar ~1 min)...")
tsne    = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42)
tsne_2d = tsne.fit_transform(all_feats)
for lbl, col in col_map.items():
    mask = np.array(labels) == lbl
    axes[1].scatter(tsne_2d[mask, 0], tsne_2d[mask, 1],
                    label=lbl, color=col, alpha=0.6, s=20)
axes[1].set_title("t-SNE", fontsize=11)
axes[1].set_xlabel("t-SNE 1"); axes[1].set_ylabel("t-SNE 2")
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("frechet_pca_tsne.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: frechet_pca_tsne.png")


Extrayendo features para visualización (200 muestras por clase)...
Calculando PCA...
Calculando t-SNE (puede tardar ~1 min)...
Guardado: frechet_pca_tsne.png


/tmp/ipykernel_213802/3793329038.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [39]:
# ── Cargar baseline ──────────────────────────────────────────
ckpt_base  = torch.load("training_logs/baseline.pt", map_location="cpu")
G_baseline = FNOGenerator(hidden_ch=64, modes1=12, modes2=12, n_layers=4, z_dim=4)
G_baseline.load_state_dict(ckpt_base["G"])
G_baseline.eval()
print(f"G_baseline cargado — época {ckpt_base['epoch']}")

# ── Extraer features: 3 modelos ──────────────────────────────
print("Extrayendo features (200 muestras por clase)...")
feat_gt       = extract_features_random(extractor, val_ds, n_samples=200)
feat_gen      = extract_features_random(extractor, val_ds, n_samples=200,
                                        use_generated=True, generator=G_fd)
feat_baseline = extract_features_random(extractor, val_ds, n_samples=200,
                                        use_generated=True, generator=G_baseline)

all_feats = np.concatenate([feat_gt, feat_gen, feat_baseline], axis=0)
labels    = (["Ground Truth"] * 200 +
             ["Generado"]     * 200 +
             ["Baseline"]     * 200)

col_map = {
    "Ground Truth": "tab:blue",
    "Generado":     "tab:red",
    "Baseline":     "tab:green",
}

# ── PCA ──────────────────────────────────────────────────────
print("Calculando PCA...")
pca    = PCA(n_components=2)
pca_2d = pca.fit_transform(all_feats)

fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle("Espacio de features — Ground Truth vs Generado vs Baseline",
             fontsize=12)

for lbl, col in col_map.items():
    mask = np.array(labels) == lbl
    ax.scatter(pca_2d[mask, 0], pca_2d[mask, 1],
               label=lbl, color=col, alpha=0.6, s=20)

ax.set_title(
    f"PCA  (PC1={pca.explained_variance_ratio_[0]*100:.1f}%, "
    f"PC2={pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=11)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("frechet_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: frechet_pca.png")

G_baseline cargado — época 10
Extrayendo features (200 muestras por clase)...
Calculando PCA...
Guardado: frechet_pca.png


/tmp/ipykernel_213802/1244882648.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [40]:
# ── Cálculo de Fréchet Distance: GT vs GT, GT vs Gen, GT vs Baseline ────
n_fd = 100

print("Calculando GT vs GT (piso)...")
fd_gt_list = []
for i in range(10):
    fa = extract_features_random(extractor, val_ds, n_samples=n_fd)
    fb = extract_features_random(extractor, val_ds, n_samples=n_fd)
    fd_gt_list.append(frechet_distance(fa, fb))

print("Calculando GT vs Generado...")
fd_gen_list = []
for i in range(10):
    fa = extract_features_random(extractor, val_ds, n_samples=n_fd)
    fb = extract_features_random(extractor, val_ds, n_samples=n_fd,
                                 use_generated=True, generator=G_fd)
    fd_gen_list.append(frechet_distance(fa, fb))

print("Calculando GT vs Baseline...")
fd_base_list = []
for i in range(10):
    fa = extract_features_random(extractor, val_ds, n_samples=n_fd)
    fb = extract_features_random(extractor, val_ds, n_samples=n_fd,
                                 use_generated=True, generator=G_baseline)
    fd_base_list.append(frechet_distance(fa, fb))

print(f"\n{'='*50}")
print(f"FD GT vs GT:       {np.mean(fd_gt_list):.4f} ± {np.std(fd_gt_list):.4f}  ← piso")
print(f"FD GT vs Gen:      {np.mean(fd_gen_list):.4f} ± {np.std(fd_gen_list):.4f}")
print(f"FD GT vs Baseline: {np.mean(fd_base_list):.4f} ± {np.std(fd_base_list):.4f}")
print(f"{'='*50}")

Calculando GT vs GT (piso)...
Calculando GT vs Generado...
Calculando GT vs Baseline...

FD GT vs GT:       0.0023 ± 0.0005  ← piso
FD GT vs Gen:      0.2507 ± 0.0159
FD GT vs Baseline: 0.0783 ± 0.0159
